# Modelo de churn de clientes mayoristas — Online Retail II

**Autor:** Lucas Vicente Marinello De Giacomi · **Unit 17 – Data Science for Business Support** · MSMK University · Entrega 2, continuación del [project brief](project-brief.md). Se ejecuta con *Restart kernel and run all*.

In [1]:
# Configuración: librerías, semilla y parámetros del negocio
import hashlib
import os
import urllib.request
import warnings
import zipfile
from pathlib import Path

from IPython.display import display

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Gráficos interactivos en JupyterLab / VS Code (igual que en clase)
pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

SEED = 42                     # random_state en todo el notebook
rng = np.random.default_rng(SEED)

DATA_DIR = Path("data")       # rutas relativas al notebook (project/)
DATA_FILE = DATA_DIR / "online_retail_ii_min.parquet"
RAW_XLSX = DATA_DIR / "raw" / "online_retail_II.xlsx"
UCI_ZIP = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
MODEL_PATH = Path("model.joblib")

# Parámetros del proceso de negocio (project brief, §2.2 y §2.3)
HORIZON_DAYS = 90             # churn = sin compra en los 90 días siguientes
HISTORY_DAYS = 365            # cliente activo = compró en los 12 meses anteriores
CAPACITY = 150                # contactos que el equipo puede hacer al mes
RULE_DAYS = 60                # regla actual: más de 60 días sin comprar = riesgo
H1_RATIO = 1.5                # regla personalizada: días sin comprar > 1,5 × su intervalo habitual
MIN_RISK = 0.5                # lista equilibrada: solo clientes con probabilidad de churn ≥ 50 %

RESULTS = {}                  # números clave que se resumen al final
print("pandas", pd.__version__, "· scikit-learn", sklearn.__version__)

pandas 3.1.1 · scikit-learn 1.9.1


## 0. Resumen ejecutivo (Executive summary)

- **Pregunta:** ¿qué clientes dejarán de comprar en los próximos 90 días, para decidir a quién ofrecer cada mes los 150 descuentos que el equipo puede gestionar?
- **Modelo:** regresión logística sobre 13 indicadores de compra, elegida frente a dos ensembles porque rinde igual y no sobreajusta.
- **Mejora frente a la regla actual** (más de 60 días sin comprar), en un mes no visto: ordenando por riesgo, 119 de 150 llamadas aciertan frente a 106; en la lista recomendada (riesgo × valor), 33 % de aciertos frente a 30 % y 17,8 % de los ingresos en riesgo cubiertos frente a 16,9 %.
- **Qué hacer:** usar el modelo en un piloto de 3 meses y medir los KPI 2 y 3.
- **Principal límite:** 150 contactos para unos 2.200 clientes que se van cada trimestre; el cuello de botella es la capacidad, no el modelo.

## 1. Del brief a una tarea de machine learning (From the brief to an ML task)

| Pregunta | En este proyecto |
|---|---|
| Decisión que alimenta | A qué 150 clientes ofrecer cada mes un descuento (brief §2.2) |
| Tipo de tarea | **Clasificación binaria** (churn sí / no), usada como ranking |
| Target | `churn = 1` si un cliente con compras en los 12 meses previos **no compra en los 90 días siguientes** al corte |
| Grano | Un cliente × un corte mensual |
| Momento y horizonte | El **día 1 de cada mes**, para los 90 días siguientes |
| Features disponibles | Solo ventas y cancelaciones **anteriores** al día 1 |
| Regla actual (baseline) | Más de 60 días sin comprar = en riesgo |
| Métrica | **Precisión de la lista de 150** e **% de ingresos en riesgo que cubre**; PR-AUC para comparar modelos |
| Traducción al negocio | KPI 3 (precisión de la lista) y KPI 2 (ingresos en riesgo cubiertos) |

**Cambios respecto al brief** (no llegó feedback de la entrega 1 antes de este plazo):
- **Cortes mensuales** (día 1, de diciembre de 2010 a septiembre de 2011) en lugar de un único corte el 10/09/2011: coinciden con el refresco del dashboard y dan suficientes filas. El test es el 01/09/2011.
- **Valor neto** (ventas menos cancelaciones): con el bruto, el primer cliente de la lista tenía un único pedido de 77.184 £ cancelado entero.
- **H1 se mide con días distintos de compra**, para evitar intervalos de 0 días.

**Por qué estas herramientas (P5).** Es una decisión mensual, con capacidad limitada y datos que ya existen: pandas reconstruye la tabla de clientes, un clasificador de scikit-learn la convierte en probabilidades y joblib guarda el pipeline para el bloque 4.

## 2. Datos y calidad (Data and quality)

Online Retail II (Chen, 2012; UCI, CC BY 4.0): 1.067.371 líneas de factura de una tienda online británica, del 01/12/2009 al 09/12/2011. El repositorio incluye `data/online_retail_ii_min.parquet` (el dataset sin `Description`); si falta, se reconstruye desde el Excel de UCI.

In [2]:
def build_data_file():
    """Crea DATA_FILE desde el Excel de UCI (lo descarga si hace falta)."""
    if not RAW_XLSX.exists():
        RAW_XLSX.parent.mkdir(parents=True, exist_ok=True)
        zip_path = RAW_XLSX.parent / "online_retail_ii.zip"
        print("Descargando el dataset desde UCI…")
        urllib.request.urlretrieve(UCI_ZIP, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(RAW_XLSX.parent)
    print("Leyendo el Excel (tarda 1–2 minutos)…")
    sheets = pd.read_excel(RAW_XLSX, sheet_name=None)
    df = pd.concat(sheets.values(), ignore_index=True).drop(columns=["Description"])
    for col in ["Invoice", "StockCode", "Country"]:
        df[col] = df[col].astype("string")
    df["Customer ID"] = df["Customer ID"].astype("Int64").astype("string")
    DATA_DIR.mkdir(exist_ok=True)
    df.to_parquet(DATA_FILE, index=False, compression="zstd")
    return df


raw = pd.read_parquet(DATA_FILE) if DATA_FILE.exists() else build_data_file()
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"])
print(f"{len(raw):,} filas × {raw.shape[1]} columnas · "
      f"{raw['InvoiceDate'].min():%d/%m/%Y} → {raw['InvoiceDate'].max():%d/%m/%Y}")
raw.head()

1,067,371 filas × 7 columnas · 01/12/2009 → 09/12/2011


,Invoice,StockCode,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


**Limpieza (L03):** (1) duplicados exactos; (2) líneas sin Customer ID; (3) cancelaciones separadas, que se usan como señal (H3) y para el valor neto; (4) ajustes (cantidad o precio ≤ 0 que no son cancelaciones); (5) **pseudonimización** del Customer ID con SHA-256 y sal.

In [3]:
log = []
def log_step(name, before, after):
    log.append({"paso": name, "filas antes": before, "filas después": after, "eliminadas": before - after,
                "% eliminado": round(100 * (before - after) / before, 1)})

df = raw.copy()
n = len(df); df = df.drop_duplicates(); log_step("1. Duplicados exactos", n, len(df))
n = len(df); df = df[df["Customer ID"].notna()]; log_step("2. Sin Customer ID", n, len(df))
df["is_cancel"] = df["Invoice"].str.startswith("C").astype(bool)
n = len(df)
adjust = (~df["is_cancel"]) & ((df["Quantity"] <= 0) | (df["Price"] <= 0))
df = df[~adjust]; log_step("4. Ajustes (cantidad o precio ≤ 0, no cancelación)", n, len(df))
df["Revenue"] = df["Quantity"] * df["Price"]

# 5. Pseudonimización con sal
SALT = os.environ.get("RETAIL_SALT", "unit17-demo-salt")
ids = df["Customer ID"].unique()
key_map = {c: hashlib.sha256((SALT + str(c)).encode("utf-8")).hexdigest()[:16] for c in ids}
df["customer_key"] = df["Customer ID"].map(key_map)
df = df.drop(columns=["Customer ID"])

sales = df[~df["is_cancel"]].copy()
cancels = df[df["is_cancel"]].copy()
cleaning_log = pd.DataFrame(log)
RESULTS["filas_brutas"] = len(raw)
RESULTS["filas_limpias"] = len(df)
RESULTS["lineas_venta"] = len(sales)
RESULTS["lineas_cancelacion"] = len(cancels)
RESULTS["clientes"] = df["customer_key"].nunique()
print(f"Ventas: {len(sales):,} líneas · Cancelaciones: {len(cancels):,} líneas · "
      f"Clientes: {df['customer_key'].nunique():,}")
cleaning_log

Ventas: 779,423 líneas · Cancelaciones: 18,390 líneas · Clientes: 5,939


,paso,filas antes,filas después,eliminadas,% eliminado
0,1. Duplicados exactos,1067371,1033034,34337,3.2
1,2. Sin Customer ID,1033034,797883,235151,22.8
2,"4. Ajustes (cantidad o precio ≤ 0, no cancelac...",797883,797813,70,0.0


**Qué significa para el jefe de cuentas.** Quedan 797.813 líneas útiles (779.423 ventas y 18.390 cancelaciones) de 5.939 clientes. Una de cada cuatro líneas no tiene cliente: esas ventas nunca podrán entrar en una lista.

### Tabla de clientes por corte

Una "foto" de los clientes activos cada día 1, con datos anteriores a esa fecha, y la etiqueta de los 90 días siguientes.

In [4]:
def build_snapshot(sales, cancels, cutoff, horizon=HORIZON_DAYS, history=HISTORY_DAYS):
    """Una fila por cliente activo en `cutoff`: features del pasado + etiqueta del futuro."""
    cutoff = pd.Timestamp(cutoff)
    start = cutoff - pd.Timedelta(days=history)
    past = sales[sales["InvoiceDate"] < cutoff]
    hist = past[past["InvoiceDate"] >= start]

    inv = (hist.groupby(["customer_key", "Invoice"])
               .agg(date=("InvoiceDate", "min"), value=("Revenue", "sum"))
               .reset_index())
    g = inv.groupby("customer_key")
    snap = pd.DataFrame({
        "last_date": g["date"].max(),
        "n_invoices_12m": g["Invoice"].nunique(),
        "revenue_12m": g["value"].sum(),
    })
    # Ritmo de compra (H1): intervalo mediano entre días distintos de compra
    days = inv.assign(day=inv["date"].dt.normalize())[["customer_key", "day"]].drop_duplicates()
    days = days.sort_values(["customer_key", "day"])
    days["gap"] = days.groupby("customer_key")["day"].diff().dt.days
    snap["n_purchase_days_12m"] = days.groupby("customer_key")["day"].nunique()
    snap["median_interval_days"] = days.groupby("customer_key")["gap"].median()

    snap["recency_days"] = (cutoff - snap["last_date"]).dt.total_seconds() / 86400
    recent = inv[inv["date"] >= cutoff - pd.Timedelta(days=90)].groupby("customer_key")["value"].sum()
    snap["revenue_last_90d"] = recent.reindex(snap.index).fillna(0.0)
    snap["n_products_12m"] = hist.groupby("customer_key")["StockCode"].nunique()
    first = past.groupby("customer_key")["InvoiceDate"].min()
    snap["tenure_days"] = (cutoff - first.reindex(snap.index)).dt.total_seconds() / 86400

    # Cancelaciones (H3)
    c = cancels[(cancels["InvoiceDate"] < cutoff) & (cancels["InvoiceDate"] >= start)]
    cg = c.groupby("customer_key")
    snap["n_cancel_12m"] = cg["Invoice"].nunique().reindex(snap.index).fillna(0)
    snap["cancel_value_12m"] = (-cg["Revenue"].sum()).reindex(snap.index).fillna(0.0)
    snap["country"] = hist.groupby("customer_key")["Country"].agg(lambda s: s.value_counts().index[0]).astype(str)

    # Features derivadas
    snap["recency_ratio"] = snap["recency_days"] / snap["median_interval_days"]
    snap["avg_basket_value"] = snap["revenue_12m"] / snap["n_invoices_12m"]
    snap["trend_90d"] = snap["revenue_last_90d"] / (snap["revenue_12m"] / 4)
    snap["has_cancel"] = (snap["n_cancel_12m"] > 0).astype(int)
    # Valor real del cliente: ventas menos cancelaciones (un pedido enorme cancelado no es valor)
    snap["net_revenue_12m"] = (snap["revenue_12m"] - snap["cancel_value_12m"]).clip(lower=0)

    # Etiqueta: ¿compra en los próximos 90 días?
    fut = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.Timedelta(days=horizon))]
    snap["churn"] = (~snap.index.isin(fut["customer_key"].unique())).astype(int)
    snap["cutoff"] = cutoff
    return snap.drop(columns="last_date").reset_index()


CUTOFFS = pd.date_range("2010-12-01", "2011-09-01", freq="MS")
snapshots = pd.concat([build_snapshot(sales, cancels, c) for c in CUTOFFS], ignore_index=True)

# Particiones en el tiempo (ver §5): la etiqueta de entrenamiento debe conocerse antes del corte evaluado
TEST_CUTOFF = pd.Timestamp("2011-09-01")
VAL_CUTOFF = pd.Timestamp("2011-06-01")
def label_known_before(c, when):
    return c + pd.Timedelta(days=HORIZON_DAYS) <= when

dev = snapshots[snapshots["cutoff"].map(lambda c: label_known_before(c, TEST_CUTOFF))].copy()
test = snapshots[snapshots["cutoff"] == TEST_CUTOFF].copy()
train = dev[dev["cutoff"].map(lambda c: label_known_before(c, VAL_CUTOFF))].copy()
val = dev[dev["cutoff"] == VAL_CUTOFF].copy()

print(f"Filas cliente × mes: {len(snapshots):,}")
pd.DataFrame({
    "conjunto": ["entrenamiento (§6)", "validación (§6)", "desarrollo = CV (§8)", "test (§8, una sola vez)"],
    "cortes": [f"{train.cutoff.min():%m/%Y} – {train.cutoff.max():%m/%Y}", f"{VAL_CUTOFF:%m/%Y}",
               f"{dev.cutoff.min():%m/%Y} – {dev.cutoff.max():%m/%Y}", f"{TEST_CUTOFF:%m/%Y}"],
    "filas": [len(train), len(val), len(dev), len(test)],
    "% churn": [round(100 * d["churn"].mean(), 1) for d in (train, val, dev, test)],
})

Filas cliente × mes: 43,010


,conjunto,cortes,filas,% churn
0,entrenamiento (§6),12/2010 – 03/2011,17010,65.7
1,validación (§6),06/2011,4338,63.4
2,desarrollo = CV (§8),12/2010 – 06/2011,30012,64.2
3,"test (§8, una sola vez)",09/2011,4335,50.5


## 3. Qué mostró la exploración (What the exploration showed)

Solo se usan los cortes de desarrollo; el test no se mira hasta §8.

In [5]:
by_cut = dev.groupby("cutoff").agg(clientes=("churn", "size"), churn=("churn", "mean")).reset_index()
by_cut["churn_pct"] = 100 * by_cut["churn"]
RESULTS["churn_medio_dev_pct"] = round(100 * dev["churn"].mean(), 1)
fig = px.bar(by_cut, x="cutoff", y="churn_pct", text=by_cut["churn_pct"].round(1),
             title="El target: % de clientes activos que no vuelven a comprar en 90 días, por mes de corte",
             labels={"cutoff": "Fecha de corte (día 1 del mes)", "churn_pct": "Churn (% de clientes activos)"})
fig.show()
by_cut

,cutoff,clientes,churn,churn_pct
0,2010-12-01,4266,0.669245,66.924519
1,2011-01-01,4231,0.670763,67.076341
2,2011-02-01,4237,0.654945,65.494454
3,2011-03-01,4276,0.632133,63.213283
4,2011-04-01,4332,0.623038,62.303786
5,2011-05-01,4332,0.613343,61.334257
6,2011-06-01,4338,0.634163,63.416321


In [6]:
# H1 — ritmo de compra: clientes con al menos 3 días de compra en los 12 meses previos (último corte de desarrollo)
last_dev = dev[dev["cutoff"] == dev["cutoff"].max()]
h1 = last_dev[last_dev["n_purchase_days_12m"] >= 3]
h1_share = (h1["median_interval_days"] > RULE_DAYS).mean()

def rule_quality(d, flag):
    tp = (flag & (d["churn"] == 1)).sum()
    return {"clientes marcados": int(flag.sum()), "precisión %": round(100 * tp / max(flag.sum(), 1), 1),
            "recall %": round(100 * tp / max(d["churn"].sum(), 1), 1)}

fixed_flag = dev["recency_days"] > RULE_DAYS
pers_flag = np.where(dev["recency_ratio"].notna(), dev["recency_ratio"] > H1_RATIO, fixed_flag).astype(bool)
h1_rules = pd.DataFrame([{"regla": f"Fija: > {RULE_DAYS} días sin comprar", **rule_quality(dev, fixed_flag)},
                         {"regla": f"Personalizada: > {H1_RATIO} × su intervalo habitual", **rule_quality(dev, pers_flag)}])
RESULTS["H1_pct_intervalo_mayor_60"] = round(100 * h1_share, 1)
RESULTS["H1_reglas"] = h1_rules.to_dict("records")

fig = px.histogram(h1, x="median_interval_days", nbins=40,
                   title=f"H1 · Intervalo habitual entre compras ({100*h1_share:.0f} % de clientes por encima de {RULE_DAYS} días)",
                   labels={"median_interval_days": "Intervalo mediano entre compras (días)", "count": "Clientes"})
fig.add_vline(x=RULE_DAYS, line_dash="dash", annotation_text=f"Regla actual: {RULE_DAYS} días")
fig.update_layout(yaxis_title="Clientes")
fig.show()
h1_rules

,regla,clientes marcados,precisión %,recall %
0,Fija: > 60 días sin comprar,18711,75.9,73.7
1,Personalizada: > 1.5 × su intervalo habitual,16472,74.9,64.0


In [7]:
# H2 — concentración de ingresos (curva de Pareto, último corte de desarrollo)
rev = last_dev["net_revenue_12m"].sort_values(ascending=False).to_numpy()
cum = np.cumsum(rev) / rev.sum()
pct_clients = np.arange(1, len(rev) + 1) / len(rev)
top20_share = cum[int(np.ceil(0.2 * len(rev))) - 1]
RESULTS["H2_pct_ingresos_top20"] = round(100 * top20_share, 1)

fig = go.Figure(go.Scatter(x=100 * pct_clients, y=100 * cum, mode="lines", name="Ingresos acumulados"))
fig.add_vline(x=20, line_dash="dash", annotation_text=f"20 % de clientes → {100*top20_share:.0f} % de ingresos")
fig.update_layout(title="H2 · Curva de Pareto: pocos clientes concentran los ingresos",
                  xaxis_title="Clientes ordenados de mayor a menor ingreso (% acumulado)",
                  yaxis_title="Ingresos netos de los últimos 12 meses (% acumulado, £)")
fig.show()

In [8]:
# H3 — churn con y sin cancelaciones, controlando por frecuencia de compra (posible confusor, L04)
dev["freq_band"] = pd.cut(dev["n_invoices_12m"], bins=[0, 1, 4, np.inf], labels=["1 factura", "2–4 facturas", "5+ facturas"])
h3 = (dev.groupby(["freq_band", "has_cancel"], observed=True)["churn"]
         .agg(clientes="size", churn="mean").reset_index())
h3["churn_pct"] = 100 * h3["churn"]
h3["grupo"] = np.where(h3["has_cancel"] == 1, "Con cancelaciones", "Sin cancelaciones")
overall = dev.groupby("has_cancel")["churn"].mean()
h3_diff = 100 * (overall.get(1, np.nan) - overall.get(0, np.nan))
band_diff = (h3.pivot(index="freq_band", columns="has_cancel", values="churn_pct")
               .pipe(lambda t: t.get(1) - t.get(0)))
RESULTS["H3_diferencia_puntos"] = round(float(h3_diff), 1)
RESULTS["H3_diferencia_por_banda"] = band_diff.round(1).to_dict()

fig = px.bar(h3, x="freq_band", y="churn_pct", color="grupo", barmode="group", text=h3["churn_pct"].round(1),
             title=f"H3 · Churn con y sin cancelaciones (diferencia global: {h3_diff:+.1f} puntos)",
             labels={"freq_band": "Frecuencia de compra (facturas en 12 meses)", "churn_pct": "Churn (% de clientes)", "grupo": ""})
fig.show()
h3[["freq_band", "grupo", "clientes", "churn_pct"]]

,freq_band,grupo,clientes,churn_pct
0,1 factura,Sin cancelaciones,8750,85.177143
1,1 factura,Con cancelaciones,1684,83.194774
2,2–4 facturas,Sin cancelaciones,7507,70.800586
3,2–4 facturas,Con cancelaciones,4321,63.897246
4,5+ facturas,Sin cancelaciones,2077,38.517092
5,5+ facturas,Con cancelaciones,5673,27.287150


In [9]:
# Tabla de hipótesis del brief con su veredicto
pers, fixed = h1_rules.iloc[1], h1_rules.iloc[0]
h1_ok = (h1_share > 0.30) and (pers["precisión %"] > fixed["precisión %"])
h2_ok = top20_share > 0.70
h3_ok = (h3_diff >= 10) and bool((band_diff.dropna() > 0).all())
hyp = pd.DataFrame([
    {"hipótesis": "H1 · La regla fija de 60 días no sirve para todos (ritmo propio)",
     "evidencia": f"{100*h1_share:.1f} % de clientes con intervalo > {RULE_DAYS} días; precisión regla personalizada "
                  f"{pers['precisión %']} % vs fija {fixed['precisión %']} %",
     "veredicto": "apoyada" if h1_ok else ("parcialmente apoyada" if (h1_share > 0.30) or (pers["precisión %"] > fixed["precisión %"]) else "refutada")},
    {"hipótesis": "H2 · El 20 % de clientes genera > 70 % de los ingresos",
     "evidencia": f"El 20 % superior genera el {100*top20_share:.1f} % de los ingresos",
     "veredicto": "apoyada" if h2_ok else "refutada"},
    {"hipótesis": "H3 · Con cancelaciones, ≥ 10 puntos más de churn",
     "evidencia": f"Diferencia global {h3_diff:+.1f} puntos; por frecuencia: "
                  + ", ".join(f"{k}: {v:+.1f}" for k, v in band_diff.dropna().items()),
     "veredicto": "apoyada" if h3_ok else ("parcialmente apoyada" if h3_diff > 0 else "refutada")},
])
RESULTS["hipotesis"] = hyp.to_dict("records")
hyp

,hipótesis,evidencia,veredicto
0,H1 · La regla fija de 60 días no sirve para to...,31.9 % de clientes con intervalo > 60 días; pr...,parcialmente apoyada
1,H2 · El 20 % de clientes genera > 70 % de los ...,El 20 % superior genera el 73.0 % de los ingresos,apoyada
2,"H3 · Con cancelaciones, ≥ 10 puntos más de churn",Diferencia global -25.1 puntos; por frecuencia...,refutada


**Hipótesis del brief**

- **H1 · parcialmente apoyada.** El 31,9 % de los clientes compra con intervalos de más de 60 días, pero la regla personalizada **no mejora** a la fija (precisión 74,9 % frente a 75,9 %; recall 64,0 % frente a 73,7 %).
- **H2 · apoyada.** El 20 % de los clientes genera el 73,0 % de los ingresos netos.
- **H3 · refutada, y al revés.** Con cancelaciones hay **25,1 puntos menos** de churn, también dentro de cada nivel de frecuencia (−2,0, −6,9 y −11,2): cancelar es señal de un cliente activo.

**Qué significa para el jefe de cuentas.** Dos de cada tres clientes activos no vuelven en 90 días (64,2 %), así que "llamar a quien lleva tiempo sin comprar" no alcanza. Lo que importa es el valor, que está muy concentrado, y una cancelación no es motivo para llamar.

## 4. Features (Features)

| Feature | Por qué debería ayudar |
|---|---|
| `recency_days` (días desde la última compra) | Base de la regla actual |
| `n_invoices_12m`, `n_products_12m` | Compromiso con la tienda |
| `revenue_12m`, `revenue_last_90d`, `avg_basket_value` (£) | Valor (H2) y tipo de cliente |
| `trend_90d` (ingresos de 90 días ÷ ingresos de 12 meses ÷ 4) | < 1 = compra menos que su media |
| `median_interval_days`, `recency_ratio` (recencia ÷ intervalo) | Ritmo propio (H1) |
| `tenure_days` | Antigüedad de la relación |
| `n_cancel_12m`, `cancel_value_12m`, `has_cancel` | Insatisfacción (H3) |
| `country` (los países pequeños se agrupan) | Diferencias de mercado |

**Leakage:** todas las features se calculan con `InvoiceDate < corte`; la etiqueta solo mira el futuro; un corte entra en entrenamiento solo si su etiqueta ya se conocía en la fecha evaluada (§5); el preprocesado se ajusta **dentro del pipeline**, solo con entrenamiento.

In [10]:
NUM_FEATURES = ["recency_days", "n_invoices_12m", "revenue_12m", "revenue_last_90d", "trend_90d",
                "median_interval_days", "recency_ratio", "avg_basket_value", "n_products_12m",
                "tenure_days", "n_cancel_12m", "cancel_value_12m", "has_cancel"]
CAT_FEATURES = ["country"]
H1_FEATURES = ["median_interval_days", "recency_ratio"]
H3_FEATURES = ["n_cancel_12m", "cancel_value_12m", "has_cancel"]
TARGET = "churn"


def make_pipeline(model, num=NUM_FEATURES, cat=CAT_FEATURES):
    """Pipeline completo: columnas de la tabla de clientes → probabilidad de churn."""
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),   # intervalo vacío si compró un solo día
        ("log", FunctionTransformer(np.log1p)),                             # columnas muy asimétricas (L05)
        ("scale", StandardScaler()),
    ])
    categorical = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=200, sparse_output=False)
    prep = ColumnTransformer([("num", numeric, list(num)), ("cat", categorical, list(cat))])
    return Pipeline([("prep", prep), ("model", clone(model))])   # clone: cada pipeline con su propio modelo


dev[NUM_FEATURES + CAT_FEATURES].describe(include="all").T[["count", "mean", "50%", "min", "max"]].round(1)

,count,mean,50%,min,max
recency_days,30012.0,115.363263,89.568056,0.170139,364.622917
n_invoices_12m,30012.0,4.23324,2.0,1.0,201.0
revenue_12m,30012.0,1935.417581,665.22,2.9,321329.74
revenue_last_90d,30012.0,479.901321,9.16,0.0,86391.77
trend_90d,30012.0,1.032217,0.047284,0.0,4.0
median_interval_days,19022.0,68.163994,49.5,1.0,364.0
recency_ratio,19022.0,3.318253,1.066626,0.001106,362.455556
avg_basket_value,30012.0,384.979644,286.75,2.9,38662.955
n_products_12m,30012.0,60.880048,37.0,1.0,1716.0
tenure_days,30012.0,282.760092,296.507639,0.170833,546.677083


## 5. Partición y baseline (Split and baseline)

Un corte entra en entrenamiento solo si `corte + 90 días ≤ fecha evaluada`; por eso julio y agosto de 2011 no se usan. **Entrenamiento:** de diciembre de 2010 a marzo de 2011 · **Validación:** junio de 2011 · **Desarrollo (CV):** de diciembre a junio · **Test:** 01/09/2011, una sola vez.

**Baselines:** *ingenuo* (equivale a elegir al azar), *regla actual* (más de 60 días) y *regla personalizada* (H1: más de 1,5 × su intervalo habitual). Cada método se evalúa con tres listas de 150: **solo riesgo**, **riesgo × valor** y **equilibrada** (riesgo ≥ 50 %, ordenada por valor).

In [11]:
def evaluate(d, score, name):
    """Métricas de ranking y de las listas de CAPACITY clientes."""
    y = d[TARGET].to_numpy()
    value = d["net_revenue_12m"].to_numpy()
    score = np.asarray(score, dtype=float)
    s = score + np.random.default_rng(SEED).random(len(score)) * 1e-9      # desempate reproducible
    churn_value = value[y == 1].sum()
    lists = {
        "riesgo": np.argsort(-s, kind="stable")[:CAPACITY],
        "riesgo×valor": np.argsort(-(s * value), kind="stable")[:CAPACITY],
        "equilibrada": np.argsort(-np.where(s >= MIN_RISK, value, -1.0), kind="stable")[:CAPACITY],
    }
    out = {"modelo": name, "ROC-AUC": round(roc_auc_score(y, s), 3), "PR-AUC": round(average_precision_score(y, s), 3)}
    for lname, idx in lists.items():
        if lname == "equilibrada":
            idx = idx[s[idx] >= MIN_RISK]
        out[f"precisión % ({lname})"] = round(100 * y[idx].mean(), 1) if len(idx) else np.nan
        out[f"ingresos cubiertos % ({lname})"] = round(100 * value[idx][y[idx] == 1].sum() / churn_value, 1)
    return out


def rule_scores(d, kind):
    """Puntuación de las reglas: 1 = marcado; el desempate ordena dentro de los marcados."""
    if kind == "fixed":
        return (d["recency_days"] > RULE_DAYS).astype(float) + d["recency_days"] / 1e6
    ratio = d["recency_ratio"]
    flag = np.where(ratio.notna(), ratio > H1_RATIO, d["recency_days"] > RULE_DAYS)
    return flag.astype(float) + ratio.fillna(d["recency_days"] / RULE_DAYS) / 1e6


naive = make_pipeline(DummyClassifier(strategy="prior")).fit(train[NUM_FEATURES + CAT_FEATURES], train[TARGET])
baselines = pd.DataFrame([
    evaluate(val, naive.predict_proba(val[NUM_FEATURES + CAT_FEATURES])[:, 1], "Baseline ingenuo (al azar)"),
    evaluate(val, rule_scores(val, "fixed"), f"Regla actual (> {RULE_DAYS} días)"),
    evaluate(val, rule_scores(val, "personal"), "Regla personalizada (H1)"),
])
baselines

,modelo,ROC-AUC,PR-AUC,precisión % (riesgo),ingresos cubiertos % (riesgo),precisión % (riesgo×valor),ingresos cubiertos % (riesgo×valor),precisión % (equilibrada),ingresos cubiertos % (equilibrada)
0,Baseline ingenuo (al azar),0.493,0.642,72.0,3.2,13.3,12.5,13.3,12.5
1,Regla actual (> 60 días),0.747,0.818,88.0,1.8,40.0,16.6,40.0,16.6
2,Regla personalizada (H1),0.683,0.767,80.0,6.6,29.3,13.6,29.3,13.6


## 6. Modelos candidatos (Candidate models)

Uno simple (regresión logística) y dos ensembles (random forest y gradient boosting), con el mismo pipeline. El PR-AUC de entrenamiento sirve para detectar sobreajuste.

In [12]:
X_COLS = NUM_FEATURES + CAT_FEATURES
MODELS = {
    "Regresión logística": LogisticRegression(max_iter=2000, random_state=SEED),
    "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=SEED),
    "Gradient boosting": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=SEED),
}
fitted, rows = {}, []
for name, model in MODELS.items():
    pipe = make_pipeline(model).fit(train[X_COLS], train[TARGET])
    fitted[name] = pipe
    r = evaluate(val, pipe.predict_proba(val[X_COLS])[:, 1], name)
    r["PR-AUC entrenamiento"] = round(average_precision_score(train[TARGET], pipe.predict_proba(train[X_COLS])[:, 1]), 3)
    rows.append(r)

comparison = pd.concat([baselines, pd.DataFrame(rows)], ignore_index=True)
RESULTS["comparacion_validacion"] = comparison.to_dict("records")

fig = px.bar(comparison, x="modelo", y="precisión % (riesgo)", text="precisión % (riesgo)",
             title=f"De {CAPACITY} llamadas, ¿qué % va a clientes que realmente se iban? (validación, junio de 2011)",
             labels={"modelo": "", "precisión % (riesgo)": f"Precisión en los {CAPACITY} de más riesgo (%)"})
fig.show()
comparison

,modelo,ROC-AUC,PR-AUC,precisión % (riesgo),ingresos cubiertos % (riesgo),precisión % (riesgo×valor),ingresos cubiertos % (riesgo×valor),precisión % (equilibrada),ingresos cubiertos % (equilibrada),PR-AUC entrenamiento
0,Baseline ingenuo (al azar),0.493,0.642,72.0,3.2,13.3,12.5,13.3,12.5,NaN
1,Regla actual (> 60 días),0.747,0.818,88.0,1.8,40.0,16.6,40.0,16.6,NaN
2,Regla personalizada (H1),0.683,0.767,80.0,6.6,29.3,13.6,29.3,13.6,NaN
3,Regresión logística,0.798,0.843,87.3,0.6,43.3,18.8,54.0,13.9,0.85
4,Random forest,0.801,0.850,92.0,0.6,42.7,19.4,64.0,16.9,0.93
5,Gradient boosting,0.785,0.841,92.7,3.6,50.0,18.2,62.0,17.3,0.95


In [13]:
# ¿Ayudan las features de H1 y H3? Prueba con y sin ellas (mismo modelo, misma validación)
ablation = []
for label, drop in [("Todas las features", []), ("Sin features de H1 (ritmo)", H1_FEATURES),
                    ("Sin features de H3 (cancelaciones)", H3_FEATURES)]:
    num = [c for c in NUM_FEATURES if c not in drop]
    for name in ["Regresión logística", "Gradient boosting"]:
        pipe = make_pipeline(MODELS[name], num=num).fit(train[num + CAT_FEATURES], train[TARGET])
        r = evaluate(val, pipe.predict_proba(val[num + CAT_FEATURES])[:, 1], name)
        ablation.append({"variante": label, **r})
ablation = pd.DataFrame(ablation)
RESULTS["ablation"] = ablation.to_dict("records")
ablation

,variante,modelo,ROC-AUC,PR-AUC,precisión % (riesgo),ingresos cubiertos % (riesgo),precisión % (riesgo×valor),ingresos cubiertos % (riesgo×valor),precisión % (equilibrada),ingresos cubiertos % (equilibrada)
0,Todas las features,Regresión logística,0.798,0.843,87.3,0.6,43.3,18.8,54.0,13.9
1,Todas las features,Gradient boosting,0.785,0.841,92.7,3.6,50.0,18.2,62.0,17.3
2,Sin features de H1 (ritmo),Regresión logística,0.795,0.846,92.0,0.4,38.7,16.1,54.0,12.5
3,Sin features de H1 (ritmo),Gradient boosting,0.782,0.838,92.7,1.6,49.3,18.1,56.7,15.0
4,Sin features de H3 (cancelaciones),Regresión logística,0.796,0.842,88.0,0.6,43.3,18.9,52.7,13.0
5,Sin features de H3 (cancelaciones),Gradient boosting,0.785,0.839,93.3,3.9,47.3,18.5,56.7,16.2


In [14]:
# Elección: el mejor PR-AUC en validación; si el modelo simple está a menos de 0,01, se prefiere el simple
scores = {r["modelo"]: r["PR-AUC"] for r in rows}
best = max(scores, key=scores.get)
CHOSEN = "Regresión logística" if scores["Regresión logística"] >= scores[best] - 0.01 else best
RESULTS["modelo_elegido"] = CHOSEN
print(f"Mejor PR-AUC en validación: {best} ({scores[best]}) · Modelo elegido: {CHOSEN}")

Mejor PR-AUC en validación: Random forest (0.85) · Modelo elegido: Regresión logística


**Lectura.** Los tres superan a la regla actual (PR-AUC 0,841–0,850 frente a 0,818). Los ensembles **sobreajustan** (0,93–0,95 en entrenamiento frente a 0,84–0,85 en validación) y la logística no (0,85 frente a 0,843). Como queda a menos de 0,01 del mejor, se elige el **modelo más simple adecuado**.

**Con y sin features:** quitar las de H1 (0,846) o las de H3 (0,842) no cambia el resultado (0,843): lo que miden ya lo recogen la frecuencia y la recencia. Es un callejón sin salida, documentado como tal.

**Qué significa para el jefe de cuentas.** Un modelo sencillo ordena mejor que la regla de los 60 días. Los modelos complejos memorizan el pasado y no aportan nada con clientes nuevos.

## 7. Métrica y coste de los errores (Metric and cost of errors)

El equipo llama a los 150 primeros, así que el umbral es la capacidad. **Verdadero positivo:** llamamos y se iba. **Falso positivo:** descuento regalado. **Falso negativo:** ingresos perdidos. Con capacidad fija, cada falso positivo ocupa el lugar de alguien que sí se iba: por eso se miden la **precisión de la lista** (KPI 3) y los **ingresos en riesgo cubiertos** (KPI 2).

In [15]:
chosen_val = fitted[CHOSEN]
p_val = chosen_val.predict_proba(val[X_COLS])[:, 1]
order = np.argsort(-(p_val + np.random.default_rng(SEED).random(len(p_val)) * 1e-9))
contacted = np.zeros(len(val), dtype=int); contacted[order[:CAPACITY]] = 1
cm = confusion_matrix(val[TARGET], contacted, labels=[1, 0])
cm_df = pd.DataFrame(cm, index=["Se iba (churn)", "Volvía igual"], columns=["Llamado (top 150)", "No llamado"])
RESULTS["matriz_confusion_val"] = cm_df.to_dict()

fig = px.imshow(cm_df, text_auto=True, color_continuous_scale="Blues",
                title=f"Validación: matriz de confusión de la lista de {CAPACITY} clientes ({CHOSEN})",
                labels={"x": "Decisión del equipo", "y": "Lo que pasó en 90 días", "color": "Clientes"})
fig.show()

# ¿Cómo cambia la precisión si la capacidad fuera otra?
ks = [50, 100, 150, 200, 300, 400]
y_val = val[TARGET].to_numpy()
fixed_order = np.argsort(-rule_scores(val, "fixed").to_numpy())
curve = pd.DataFrame({
    "contactos": ks * 2,
    "precisión %": [100 * y_val[order[:k]].mean() for k in ks] + [100 * y_val[fixed_order[:k]].mean() for k in ks],
    "método": [CHOSEN] * len(ks) + [f"Regla actual (> {RULE_DAYS} días)"] * len(ks),
})
fig = px.line(curve, x="contactos", y="precisión %", color="método", markers=True,
              title="Precisión de la lista según cuántos clientes se contactan (validación)",
              labels={"contactos": "Clientes contactados al mes", "precisión %": "Precisión (% que realmente se iban)"})
fig.show()

# Tres formas de armar la lista de 150 con el mismo modelo (validación): el trade-off precisión vs ingresos
strategies = pd.DataFrame([{
    "lista": lname,
    "precisión %": comparison.set_index("modelo").loc[CHOSEN, f"precisión % ({key})"],
    "ingresos en riesgo cubiertos %": comparison.set_index("modelo").loc[CHOSEN, f"ingresos cubiertos % ({key})"],
} for lname, key in [("Solo riesgo", "riesgo"), ("Riesgo × valor", "riesgo×valor"),
                     (f"Equilibrada: riesgo ≥ {int(100*MIN_RISK)} %, ordenada por valor", "equilibrada")]])
RESULTS["estrategias_lista_val"] = strategies.to_dict("records")
display(cm_df)
strategies

,Llamado (top 150),No llamado
Se iba (churn),131,2620
Volvía igual,19,1568


,lista,precisión %,ingresos en riesgo cubiertos %
0,Solo riesgo,87.3,0.6
1,Riesgo × valor,43.3,18.8
2,"Equilibrada: riesgo ≥ 50 %, ordenada por valor",54.0,13.9


**Lectura.** Por riesgo, 131 de 150 llamadas aciertan (87,3 %), pero cubren solo el 0,6 % de los ingresos en riesgo: son clientes con una sola compra pequeña. Por **riesgo × valor**, la precisión baja al 43,3 % y los ingresos cubiertos suben al 18,8 %; la lista equilibrada queda en medio (54,0 % y 13,9 %). Como el KPI principal es el de ingresos, se recomienda **riesgo × valor**.

**Qué significa para el jefe de cuentas.** Llamar solo a quien es casi seguro que se va sería llamar a clientes que casi no compran. Conviene aceptar algunas llamadas "de más" para proteger a los clientes que pesan en las ventas.

## 8. Validación y ajuste (Validation and tuning)

Validación cruzada en el tiempo: cuatro folds que validan en marzo, abril, mayo y junio de 2011, cada uno entrenado solo con etiquetas ya conocidas. `GridSearchCV` sobre `C`, optimizando PR-AUC. Una única evaluación en test.

In [16]:
dev = dev.reset_index(drop=True)
folds = []
for vc in pd.date_range("2011-03-01", VAL_CUTOFF, freq="MS"):
    tr_idx = dev.index[dev["cutoff"].map(lambda c: label_known_before(c, vc))].to_numpy()
    va_idx = dev.index[dev["cutoff"] == vc].to_numpy()
    if len(tr_idx) and len(va_idx):
        folds.append((tr_idx, va_idx))

GRIDS = {
    "Regresión logística": {"model__C": [0.01, 0.1, 1, 10]},
    "Random forest": {"model__min_samples_leaf": [10, 20, 50], "model__max_features": ["sqrt", 0.5]},
    "Gradient boosting": {"model__learning_rate": [0.03, 0.1], "model__max_depth": [3, None],
                          "model__min_samples_leaf": [20, 100]},
}
search = GridSearchCV(make_pipeline(MODELS[CHOSEN]), GRIDS[CHOSEN], cv=folds,
                      scoring="average_precision", return_train_score=True, n_jobs=-1)
search.fit(dev[X_COLS], dev[TARGET])

cv = pd.DataFrame(search.cv_results_)
best_i = search.best_index_
fold_scores = pd.DataFrame({
    "fold (corte validado)": [f"{dev.loc[va[0], 'cutoff']:%m/%Y}" for _, va in folds],
    "PR-AUC validación": [round(cv.loc[best_i, f"split{i}_test_score"], 3) for i in range(len(folds))],
    "PR-AUC entrenamiento": [round(cv.loc[best_i, f"split{i}_train_score"], 3) for i in range(len(folds))],
})
RESULTS["mejores_parametros"] = search.best_params_
RESULTS["cv_pr_auc_media"] = round(cv.loc[best_i, "mean_test_score"], 3)
RESULTS["cv_pr_auc_std"] = round(cv.loc[best_i, "std_test_score"], 3)
RESULTS["cv_pr_auc_train_media"] = round(cv.loc[best_i, "mean_train_score"], 3)
print("Mejores parámetros:", search.best_params_)
print(f"PR-AUC CV: {RESULTS['cv_pr_auc_media']} ± {RESULTS['cv_pr_auc_std']} "
      f"(entrenamiento: {RESULTS['cv_pr_auc_train_media']})")
fold_scores

Mejores parámetros: {'model__C': 10}
PR-AUC CV: 0.832 ± 0.006 (entrenamiento: 0.864)


,fold (corte validado),PR-AUC validación,PR-AUC entrenamiento
0,03/2011,0.829,0.871
1,04/2011,0.828,0.867
2,05/2011,0.827,0.867
3,06/2011,0.843,0.850


In [17]:
# Evaluación final, una sola vez, en el corte de test (01/09/2011)
final_model = search.best_estimator_          # reentrenado con todos los cortes de desarrollo
p_test = final_model.predict_proba(test[X_COLS])[:, 1]
naive_dev = make_pipeline(DummyClassifier(strategy="prior")).fit(dev[X_COLS], dev[TARGET])
test_results = pd.DataFrame([
    evaluate(test, naive_dev.predict_proba(test[X_COLS])[:, 1], "Baseline ingenuo (al azar)"),
    evaluate(test, rule_scores(test, "fixed"), f"Regla actual (> {RULE_DAYS} días)"),
    evaluate(test, rule_scores(test, "personal"), "Regla personalizada (H1)"),
    evaluate(test, p_test, f"{CHOSEN} ajustado"),
])
RESULTS["test"] = test_results.to_dict("records")
RESULTS["test_clientes"] = len(test)
RESULTS["test_churn_pct"] = round(100 * test[TARGET].mean(), 1)
test_results

,modelo,ROC-AUC,PR-AUC,precisión % (riesgo),ingresos cubiertos % (riesgo),precisión % (riesgo×valor),ingresos cubiertos % (riesgo×valor),precisión % (equilibrada),ingresos cubiertos % (equilibrada)
0,Baseline ingenuo (al azar),0.500,0.504,52.0,5.8,9.3,11.4,9.3,11.4
1,Regla actual (> 60 días),0.709,0.682,70.7,2.2,30.0,16.9,30.0,16.9
2,Regla personalizada (H1),0.654,0.622,60.7,7.1,20.7,13.7,20.7,13.7
3,Regresión logística ajustado,0.771,0.732,79.3,0.7,33.3,17.8,39.3,14.1


**Lectura.** Con `C = 10`, el PR-AUC es **0,832 ± 0,006** entre folds (estable) frente a 0,864 en entrenamiento (sin sobreajuste). En el test baja a 0,732: en septiembre el churn es menor (50,5 % frente a 64 %), porque antes de Navidad vuelven más clientes. Aun así supera a la regla en todo: PR-AUC 0,732 frente a 0,682, **precisión por riesgo 79,3 % frente a 70,7 %** y, en la lista riesgo × valor, 33,3 % frente a 30,0 % de aciertos y 17,8 % frente a 16,9 % de ingresos cubiertos.

**Qué significa para el jefe de cuentas.** En un mes no visto, el modelo pone en la lista 13 clientes correctos más de 150 si se busca solo riesgo, y 5 más en la lista por valor. La mejora es real pero modesta.

## 9. Interpretación (Interpretation)

Importancia por permutación en el test y tres clientes explicados con un "¿y si…?": cuánto cambia su probabilidad si una feature tomara el valor típico.

In [18]:
imp = permutation_importance(final_model, test[X_COLS], test[TARGET], scoring="average_precision",
                             n_repeats=10, random_state=SEED, n_jobs=-1)
importance = (pd.DataFrame({"feature": X_COLS, "importancia": imp.importances_mean, "std": imp.importances_std})
                .sort_values("importancia", ascending=True))
RESULTS["importancia_top5"] = importance.sort_values("importancia", ascending=False).head(5)[["feature", "importancia"]].round(4).to_dict("records")
fig = px.bar(importance, x="importancia", y="feature", orientation="h", error_x="std",
             title="¿Qué mira el modelo? Importancia por permutación (test)",
             labels={"importancia": "Caída del PR-AUC al desordenar la columna", "feature": ""})
fig.show()

In [19]:
# Tres predicciones individuales explicadas
test = test.reset_index(drop=True)
test["p_churn"] = p_test
test["prioridad"] = test["p_churn"] * test["net_revenue_12m"]   # lista recomendada: riesgo × valor (ingresos esperados perdidos)
ranked = test.sort_values("prioridad", ascending=False).reset_index(drop=True)
outside = ranked.iloc[CAPACITY:]
examples = {
    "Nº 1 de la lista": ranked.iloc[0],
    f"Nº {CAPACITY} (último de la lista)": ranked.iloc[CAPACITY - 1],
    "Cliente grande fuera de la lista": outside.sort_values("net_revenue_12m", ascending=False).iloc[0],
}
medians = dev[NUM_FEATURES].median()

def explain(row, top=3):
    x = row[X_COLS].to_frame().T.astype({c: float for c in NUM_FEATURES})
    base = final_model.predict_proba(x)[:, 1][0]
    effects = {}
    for f in NUM_FEATURES:
        x2 = x.copy(); x2[f] = medians[f]
        effects[f] = base - final_model.predict_proba(x2)[:, 1][0]
    main = sorted(effects.items(), key=lambda kv: -abs(kv[1]))[:top]
    return base, ", ".join(f"{f} = {row[f]:.1f} ({'+' if e > 0 else ''}{100*e:.0f} pts)" for f, e in main)

expl = []
for label, row in examples.items():
    p, why = explain(row)
    expl.append({"cliente": label, "clave": row["customer_key"][:8] + "…", "prob. churn %": round(100 * p, 1),
                 "ingresos netos 12m £": round(row["net_revenue_12m"]), "se fue de verdad": "sí" if row["churn"] else "no",
                 "qué empuja la predicción (efecto vs. valor típico)": why})
expl = pd.DataFrame(expl)
RESULTS["explicaciones"] = expl.to_dict("records")
expl

,cliente,clave,prob. churn %,ingresos netos 12m £,se fue de verdad,qué empuja la predicción (efecto vs. valor típico)
0,Nº 1 de la lista,86a92a95…,65.9,21536,sí,"revenue_12m = 44534.3 (+66 pts), avg_basket_va..."
1,Nº 150 (último de la lista),4001c7bd…,32.3,4964,sí,"n_invoices_12m = 8.0 (-64 pts), avg_basket_val..."
2,Cliente grande fuera de la lista,0b7903ca…,0.4,265182,no,"n_invoices_12m = 74.0 (-99 pts), avg_basket_va..."


**¿Coinciden con las hipótesis?** Lo que más pesa, de lejos, es el **número de facturas**; después, el valor medio del pedido y los ingresos. Las features de ritmo (H1) pesan poco y las de cancelaciones (H3) casi nada, en línea con §3. El cliente grande fuera de la lista (265.182 £ netos) compra cada 5 días y su riesgo es casi nulo: llamarlo sería regalar un descuento.

**Qué significa para el jefe de cuentas.** El modelo dice, sobre todo, "este cliente compra pocas veces al año". Los clientes con pocos pedidos son los que hay que vigilar.

## 10. Equidad, ética y riesgos (Fairness, ethics and risks)

Errores de la lista recomendada (riesgo × valor) por **país** (Reino Unido / resto) y por **valor** (20 % superior / resto).

In [20]:
in_list = np.zeros(len(test), dtype=bool)
in_list[test["prioridad"].to_numpy().argsort()[::-1][:CAPACITY]] = True
test["en_lista"] = in_list
test["grupo_pais"] = np.where(test["country"] == "United Kingdom", "Reino Unido", "Otros países")
top_value_cut = test["net_revenue_12m"].quantile(0.8)
test["grupo_valor"] = np.where(test["net_revenue_12m"] >= top_value_cut, "Alto valor (20 % superior)", "Resto")

def group_errors(col):
    out = []
    for g, d in test.groupby(col):
        churners = d["churn"] == 1
        out.append({"grupo": g, "clientes": len(d), "churn %": round(100 * churners.mean(), 1),
                    "en la lista": int(d["en_lista"].sum()),
                    "recall % (churners que la lista detecta)": round(100 * (d["en_lista"] & churners).sum() / max(churners.sum(), 1), 1),
                    "precisión % en la lista": round(100 * d.loc[d["en_lista"], "churn"].mean(), 1) if d["en_lista"].any() else np.nan})
    return pd.DataFrame(out)

fair = pd.concat([group_errors("grupo_pais").assign(dimensión="País"),
                  group_errors("grupo_valor").assign(dimensión="Valor")], ignore_index=True)
RESULTS["equidad"] = fair.to_dict("records")
fig = px.bar(fair, x="grupo", y="recall % (churners que la lista detecta)", color="dimensión",
             text="recall % (churners que la lista detecta)",
             title=f"¿A quién detecta la lista de {CAPACITY}? Recall por grupo (test)",
             labels={"grupo": "", "recall % (churners que la lista detecta)": "Churners detectados por la lista (%)"})
fig.show()
fair

,grupo,clientes,churn %,en la lista,recall % (churners que la lista detecta),precisión % en la lista,dimensión
0,Otros países,388,47.4,41,8.2,36.6,País
1,Reino Unido,3947,50.8,109,1.7,32.1,País
2,Alto valor (20 % superior),867,15.6,150,37.0,33.3,Valor
3,Resto,3468,59.2,0,0.0,NaN,Valor


**Lectura.** Por diseño, la lista se llena con clientes de alto valor, y los pequeños, que son los que más abandonan,no reciben ningún contacto (0 de 150). Por país, con solo 388 clientes no británicos en el test, las diferencias son poco fiables y hay que vigilarlas mes a mes.

**Datos personales.** No hay nombres, pero el ID, junto con el país y el historial, podría identificar a un autónomo. Se pseudonimiza antes del análisis y la correspondencia con el cliente real queda solo en el sistema del equipo de cuentas (GDPR, minimización de datos).

| Riesgo | Afecta a | Mitigación |
|---|---|---|
| Descuento a quien volvía igual | Margen | Lista limitada a 150; KPI 3 mensual |
| Cliente perdido sin llamada | Ingresos | Regla de 60 días como alerta secundaria |
| Clientes pequeños nunca priorizados | Clientes pequeños | Reservarles unos 20 contactos al mes |
| Peor detección fuera del Reino Unido | Clientes internacionales | Recall por país mensual |
| Datos de 2011 y efecto de la temporada | Todas las decisiones | Reentrenar y vigilar la deriva (bloque 4) |

**Qué significa para el jefe de cuentas.** La lista protege sobre todo los ingresos de los clientes grandes. Si la empresa quiere cuidar también a los pequeños, tiene que reservarles llamadas a propósito.

## 11. Recomendación (Recommendation)

1. **Piloto de 3 meses** ordenando la lista mensual con el modelo (riesgo × valor) en lugar de la regla: en el test, unos 5 aciertos más de 150 (33,3 % frente a 30,0 %) y el 17,8 % de los ingresos en riesgo cubiertos frente al 16,9 %.
2. **Reservar unos 20 contactos** para los clientes pequeños de mayor riesgo.
3. **Medir cada mes los KPI 2 y 3** frente a la regla; si en 3 meses el modelo no la supera, volver a la regla.
4. **Ampliar la capacidad o usar un canal más barato** (email): 150 llamadas no cubren más de una quinta parte de los ingresos en juego.

**No fiarse del modelo** con datos de otro año sin reentrenar, en temporadas que no estuvieron en el entrenamiento ni en países con pocos clientes. **Bloque 4:** servir el modelo cada día 1 (L10), vigilar la deriva y el KPI 3 (L11) y estimar el impacto económico (L12).

## 12. Guardar el modelo (Saving the model)

El pipeline completo se guarda en `project/model.joblib`. Recibe la tabla de `build_snapshot` y devuelve la probabilidad de churn.

In [21]:
joblib.dump(final_model, MODEL_PATH)
reloaded = joblib.load(MODEL_PATH)
one = test[X_COLS].iloc[[0]]
print(f"Guardado en {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e6:.2f} MB)")
print("Predicción de prueba con el modelo recargado:", round(float(reloaded.predict_proba(one)[:, 1][0]), 3))
assert np.isclose(reloaded.predict_proba(one)[:, 1][0], final_model.predict_proba(one)[:, 1][0])

import plotly
for lib in (pd, np, sklearn, joblib, plotly):
    print(f"{lib.__name__:10s} {lib.__version__}")

Guardado en model.joblib (0.01 MB)
Predicción de prueba con el modelo recargado: 0.874
pandas     3.1.1
numpy      2.5.4
sklearn    1.9.1
joblib     1.6.0
plotly     7.1.0


In [22]:
# Resumen de resultados (para redactar los textos)
import json
print(json.dumps(RESULTS, ensure_ascii=False, indent=1, default=str))
Path("results.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=1, default=str), encoding="utf-8")

{
 "filas_brutas": 1067371,
 "filas_limpias": 797813,
 "lineas_venta": 779423,
 "lineas_cancelacion": 18390,
 "clientes": 5939,
 "churn_medio_dev_pct": 64.2,
 "H1_pct_intervalo_mayor_60": 31.9,
 "H1_reglas": [
  {
   "regla": "Fija: > 60 días sin comprar",
   "clientes marcados": 18711,
   "precisión %": 75.9,
   "recall %": 73.7
  },
  {
   "regla": "Personalizada: > 1.5 × su intervalo habitual",
   "clientes marcados": 16472,
   "precisión %": 74.9,
   "recall %": 64.0
  }
 ],
 "H2_pct_ingresos_top20": 73.0,
 "H3_diferencia_puntos": -25.1,
 "H3_diferencia_por_banda": {
  "1 factura": -2.0,
  "2–4 facturas": -6.9,
  "5+ facturas": -11.2
 },
 "hipotesis": [
  {
   "hipótesis": "H1 · La regla fija de 60 días no sirve para todos (ritmo propio)",
   "evidencia": "31.9 % de clientes con intervalo > 60 días; precisión regla personalizada 74.9 % vs fija 75.9 %",
   "veredicto": "parcialmente apoyada"
  },
  {
   "hipótesis": "H2 · El 20 % de clientes genera > 70 % de los ingresos",
   "evide

10185

## 13. Referencias (References)

Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324

Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D

Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *The Annals of Statistics, 29*(5), 1189–1232. https://doi.org/10.1214/aos/1013203451

McKinney, W. (2010). Data structures for statistical computing in Python. En S. van der Walt y J. Millman (Eds.), *Proceedings of the 9th Python in Science Conference* (pp. 56–61). https://doi.org/10.25080/Majora-92bf1922-00a

Ortega, M. (2026). *BPS-Dsci: Unit 17 – Business Process Support* [Material del curso]. GitHub. https://github.com/Moises-dsci/BPS-Dsci

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.

Plotly Technologies Inc. (2015). *Collaborative data science*. https://plot.ly

Reglamento (UE) 2016/679 del Parlamento Europeo y del Consejo, de 27 de abril de 2016, relativo a la protección de las personas físicas en lo que respecta al tratamiento de datos personales (Reglamento general de protección de datos). *Diario Oficial de la Unión Europea, L 119*, 1–88.